In [31]:
import pandas as pd
pd.set_option("display.max_columns", None)


In [32]:
dados = pd.read_csv('../data/klike_challenge_dataset.csv')
dados.head(5)

,campaign_id,date,platform,category,objective,target_audience_age,is_retargeting,video_duration_s,format,has_subtitle,has_cta,has_hook,has_face,text_density,music_voice_ratio,impressions,clicks,ctr,cpc,spend,conversions,revenue,roas,avg_watch_time_s,engagement_rate,klike_score
0,KLK-0001,2024-01-01,Meta,Branding,awareness,18-24,False,NaN,vertical,True,True,False,True,medium,0.39,150825,5882,0.0390,0.82,4823.24,48,0.00,0.00,3.9,0.0446,63.6
1,KLK-0002,2024-03-22,LinkedIn,App Install,traffic,25-34,False,NaN,horizontal,True,True,True,True,low,0.07,7433,493,0.0664,3.54,1745.22,34,NaN,0.16,6.4,0.0217,70.0
2,KLK-0003,2024-05-04,TikTok,SaaS,traffic,25-34,False,8.0,vertical,True,True,True,True,medium,0.43,193198,15069,0.0780,0.38,5726.22,492,51053.44,8.92,8.0,0.1000,99.1
3,KLK-0004,2024-12-27,TikTok,Lead Gen,engagement,25-34,False,27.0,quadrado,False,True,True,True,medium,0.54,6506,601,0.0925,0.71,426.71,19,1113.78,2.61,20.3,0.0414,74.9
4,KLK-0005,2024-07-10,Meta,Branding,engagement,25-34,False,34.0,vertical,True,True,True,True,low,0.35,63830,2355,0.0369,0.38,894.90,13,0.00,0.00,34.0,0.0184,77.9


In [16]:
df_hook = dados.groupby(['platform', 'has_hook'], observed=True)['roas'].agg(['count', 'median'])
com_hook = df_hook.xs(True, level='has_hook')['median']
sem_hook = df_hook.xs(False, level='has_hook')['median']

diferenca_pct = ((com_hook - sem_hook) / sem_hook) * 100
display(diferenca_pct.round(2))
df_hook

platform
LinkedIn     0.00
Meta       -27.07
TikTok      35.91
Name: median, dtype: float64

count  median
platform has_hook               
LinkedIn False        38    0.53
         True         45    0.53
Meta     False       113    1.81
         True        128    1.32
TikTok   False        93    2.20
         True         83    2.99

In [34]:
def comparar_atributo(dados, atributo, metrica, atual, alternativa):
    df = dados.groupby(['platform', atributo], observed=True)[metrica].agg(['count', 'median'])
    grupo_atual = df.xs(atual, level=atributo)['median']
    grupo_alternativa = df.xs(alternativa, level=atributo)['median']
    diferenca_pct = ((grupo_alternativa - grupo_atual) / grupo_atual) * 100
    return diferenca_pct

def melhor_alternativa(dados, plataforma, atributo, metrica, valor_atual):
    df = dados[dados['platform'] == plataforma]
    resumo = df.groupby(atributo, observed=True)[metrica].agg(['count', 'median'])
    # Considera só grupos com pelo menos 10 campanhas
    resumo = resumo[resumo['count'] >= 10]

    if valor_atual not in resumo.index:
        return None

    mediana_atual = resumo.loc[valor_atual, 'median']
    # Remove a categoria atual das alternativas
    alternativas = resumo.drop(index=valor_atual)

    if alternativas.empty or mediana_atual <= 0:
        return None

    # Seleciona alternativa com maior mediana
    melhor_valor = alternativas['median'].idxmax()
    melhor_mediana = alternativas.loc[melhor_valor, 'median']

    diferenca_pct = (melhor_mediana - mediana_atual) / mediana_atual * 100

    if diferenca_pct <= 0:
        return None

    return {'atributo': atributo, 'valor_atual': valor_atual, 'valor_recomendado': melhor_valor, 
            'metrica': metrica, 'diferenca_pct': diferenca_pct, 'mediana_atual': mediana_atual,
            'mediana_alternativa': melhor_mediana, 'n_atual': int(resumo.loc[valor_atual, 'count']),
            'n_alternativa': int(resumo.loc[melhor_valor, 'count'])}

In [35]:
campanha = dados.iloc[4]

atributos = [
    "has_hook",
    "has_cta",
    "has_face",
    "has_subtitle",
    "text_density"
]

metricas_por_objetivo = {
    "traffic": "ctr",
    "engagement": "engagement_rate",
    "conversions": "roas",
    "awareness": "avg_watch_time_s",
    "app_install": "conversions"
}

metrica = metricas_por_objetivo[campanha["objective"]]

for atributo in atributos:
    resultado = melhor_alternativa(
        dados,
        plataforma=campanha["platform"],
        atributo=atributo,
        metrica=metrica,
        valor_atual=campanha[atributo]
    )

    if resultado is not None:
        print(resultado)

{'atributo': 'has_subtitle', 'valor_atual': True, 'valor_recomendado': np.False_, 'metrica': 'engagement_rate', 'diferenca_pct': np.float64(1.9774011299435006), 'mediana_atual': np.float64(0.0177), 'mediana_alternativa': np.float64(0.01805), 'n_atual': 119, 'n_alternativa': 86}
{'atributo': 'text_density', 'valor_atual': 'low', 'valor_recomendado': 'medium', 'metrica': 'engagement_rate', 'diferenca_pct': np.float64(23.030303030303013), 'mediana_atual': np.float64(0.0165), 'mediana_alternativa': np.float64(0.0203), 'n_atual': 97, 'n_alternativa': 96}
